# KAN-to-MLP Parameter Matching

This notebook estimates the KAN parameter range and selects three MLP configurations with comparable parameter counts.

The final objective is to recommend:

1. A 1-hidden-layer MLP matching the minimum KAN parameter budget.
2. A 2-hidden-layer MLP matching the intermediate budget.
3. A 3-hidden-layer MLP matching the maximum KAN parameter budget.

The KAN range includes all tested hidden layers, hidden units, grid sizes, and spline orders.

In [1]:
import os
import sys
from datetime import datetime

from importlib import reload
current_dir = os.getcwd()
utilities_dir = os.path.join(current_dir, '../../utils')
os.chdir(current_dir)
if utilities_dir not in sys.path:
    sys.path.insert(0, utilities_dir)
import plotting
import pinns_infinite
import infinite
reload(plotting)
reload(pinns_infinite)
reload(infinite)
import numpy as np
import sympy as sp
from calflops import calculate_flops
import matplotlib.pyplot as plt 
import torch
import torch.nn as nn
import torch.optim as optim
from pinns_infinite import  MLP, init_weights, CoefficientNet, pde_loss_inf, observation_loss_u, observation_loss_k, train_dual_network, build_models, set_seed,run_experiment_inf,build_models_KAN
from pinns_infinite import build_models
from infinite import analytical_solution_inf, coefficient_inf, source_term_inf, generate_dataset_inf, evaluate_model_inf
torch.set_default_dtype(torch.float32)
from plotting import plot_histories_comparison

set_seed(1)
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

/home/orincon/miniconda3/envs/PIKAN-unbounded-domains-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Search Spaces and Candidate Configurations

In [11]:
from itertools import product


# Activación personalizada
class Sine(nn.Module):
    def forward(self, x):
        return torch.sin(x)
# Candidate values for the KAN search.
# Five optimization hyperparameters:
# hidden_layers, hidden_units, grid_size, spline_order, learning_rate
kan_search_space = {
    "hidden_layers": [1, 2, 3],
    "hidden_units": [15, 25, 35],
    "grid_size": [3, 5, 7],
    "spline_order": [2, 3, 4],
    "learning_rate": [1e-4, 1e-3, 1e-2],
}

# Candidate values for the MLP search.
# Four optimization hyperparameters:
# hidden_layers, hidden_units, activation, learning_rate
mlp_search_space = {
    "hidden_layers": [1, 2, 3],
    "hidden_units": [15, 90, 104],
    "activation": ["Sine", "Sigmoid", "Tanh"],
    "learning_rate": [1e-4, 1e-3, 1e-2],
}

kan_configurations = [
    dict(zip(kan_search_space, values))
    for values in product(*kan_search_space.values())
]

mlp_configurations = [
    dict(zip(mlp_search_space, values))
    for values in product(*mlp_search_space.values())
]

print(f"KAN candidate configurations: {len(kan_configurations)}")
print(f"MLP candidate configurations: {len(mlp_configurations)}")

print("\nExample KAN configuration:")
print(kan_configurations[0])

print("\nExample MLP configuration:")
print(mlp_configurations[0])

KAN candidate configurations: 243
MLP candidate configurations: 81

Example KAN configuration:
{'hidden_layers': 1, 'hidden_units': 15, 'grid_size': 3, 'spline_order': 2, 'learning_rate': 0.0001}

Example MLP configuration:
{'hidden_layers': 1, 'hidden_units': 15, 'activation': 'Sine', 'learning_rate': 0.0001}


In [3]:
import json
from pathlib import Path

data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

kan_config_path = data_dir / "kan_configurations.json"
mlp_config_path = data_dir / "mlp_configurations.json"

with kan_config_path.open("w", encoding="utf-8") as file:
    json.dump(kan_configurations, file, indent=2)

with mlp_config_path.open("w", encoding="utf-8") as file:
    json.dump(mlp_configurations, file, indent=2)

print(f"Saved {len(kan_configurations)} KAN configurations to {kan_config_path}")
print(f"Saved {len(mlp_configurations)} MLP configurations to {mlp_config_path}")

Saved 243 KAN configurations to data/kan_configurations.json
Saved 81 MLP configurations to data/mlp_configurations.json


## 1. Generate KAN and MLP Architecture Candidates

The KAN sweep defines the reference parameter range. The MLP sweep provides candidate widths for the three required depths.

In [6]:
# -------------------------
# Setup & Configurations
# -------------------------
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

input_shape = (1, 2)

kan_depths = [1, 2, 3]
kan_widths = [15, 25, 35]
kan_grids = [3, 5, 7]
spline_orders = [2, 3, 4]

# MLP widths to test
mlp_sweep_widths = range(10, 150, 1)


# -------------------------
# Storage
# -------------------------
kan_labels = []
kan_depths_list = []
kan_widths_list = []
kan_grids_list = []
kan_spline_orders_list = []

kan_params_list = []
mlp_matched_params_list = []

mlp_matched_widths = []


for depth in kan_depths:
    for units in kan_widths:
        for grid in kan_grids:
            for spline_order in spline_orders:
                # -------------------------------------------------
                # Build KAN
                # -------------------------------------------------
                model_kan, _ = build_models_KAN(
                    device,
                    hidden_layers=depth,
                    hidden_units=units,
                    grid_size=grid,
                    spline_order=spline_order,
                )

                # -------------------------------------------------
                # Calculate KAN parameters
                # -------------------------------------------------
                _, _, kan_params = calculate_flops(
                    model_kan,
                    input_shape=input_shape,
                    print_results=False,
                    print_detailed=False,
                    output_as_string=False,
                )

                # -------------------------------------------------
                # Find MLP with closest number of parameters
                # -------------------------------------------------
                best_width = None
                best_mlp_params = None
                min_difference = float("inf")

                for m_width in mlp_sweep_widths:
                    model_mlp, _ = build_models(
                        device,
                        hidden_layers=depth,
                        hidden_units=m_width,
                    )

                    _, _, mlp_params = calculate_flops(
                        model_mlp,
                        input_shape=input_shape,
                        print_results=False,
                        print_detailed=False,
                        output_as_string=False,
                    )

                    difference = abs(mlp_params - kan_params)
                    if difference < min_difference:
                        min_difference = difference
                        best_width = m_width
                        best_mlp_params = mlp_params

                # -------------------------------------------------
                # Store results
                # -------------------------------------------------
                kan_labels.append(f"L={depth}, N={units}, G={grid}, k={spline_order}")
                kan_depths_list.append(depth)
                kan_widths_list.append(units)
                kan_grids_list.append(grid)
                kan_spline_orders_list.append(spline_order)
                kan_params_list.append(kan_params)
                mlp_matched_widths.append(best_width)
                mlp_matched_params_list.append(best_mlp_params)

## 2. Compare Individual KAN Architectures

This table shows the closest MLP architecture for each tested KAN configuration.

In [29]:
# ---------------------------------------------------------
# Summary table
# ---------------------------------------------------------
print("\n")
print("=" * 75)
print("PARAMETER-MATCHED ARCHITECTURES")
print("=" * 75)

print(
    f"{'KAN':<12} | "
    f"{'KAN Params':>12} | "
    f"{'MLP Width':>10} | "
    f"{'MLP Params':>12} | "
    f"{'Difference':>12}"
)

print("-" * 75)


for label, kan_p, mlp_w, mlp_p in zip(
    kan_labels,
    kan_params_list,
    mlp_matched_widths,
    mlp_matched_params_list,
):

    difference = abs(
        kan_p - mlp_p
    )

    print(
        f"{label.replace(chr(10), ', '):<12} | "
        f"{kan_p:>12,} | "
        f"{mlp_w:>10} | "
        f"{mlp_p:>12,} | "
        f"{difference:>12,}"
    )



PARAMETER-MATCHED ARCHITECTURES
KAN          |   KAN Params |  MLP Width |   MLP Params |   Difference
---------------------------------------------------------------------------
L=1, N=15    |          450 |         19 |          457 |            7
L=1, N=25    |          750 |         25 |          751 |            1
L=1, N=35    |        1,050 |         30 |        1,051 |            1
L=2, N=15    |        2,700 |         35 |        2,661 |           39
L=2, N=25    |        7,000 |         58 |        7,077 |           77
L=2, N=35    |       13,300 |         80 |       13,281 |           19
L=3, N=15    |        4,950 |         39 |        4,837 |          113
L=3, N=25    |       13,250 |         65 |       13,131 |          119
L=3, N=35    |       25,550 |         91 |       25,481 |           69


## 3. Determine the KAN Parameter Range

This is the central reference calculation. It identifies the minimum and maximum KAN parameter counts across all tested combinations and estimates the MLP width range that falls between them.

In [7]:
import pandas as pd

# KAN parameter range over all tested depths, widths, grids, and spline orders.
kan_results = pd.DataFrame({
    "hidden_layers": kan_depths_list,
    "hidden_units": kan_widths_list,
    "grid_size": kan_grids_list,
    "spline_order": kan_spline_orders_list,
    "parameters": kan_params_list,
})

kan_min_row = kan_results.loc[kan_results["parameters"].idxmin()]
kan_max_row = kan_results.loc[kan_results["parameters"].idxmax()]
kan_min_params = int(kan_min_row["parameters"])
kan_max_params = int(kan_max_row["parameters"])

print("KAN PARAMETER RANGE")
print("=" * 70)
print(
    "Minimum: "
    f"{kan_min_params:,} parameters "
    f"(L={int(kan_min_row['hidden_layers'])}, "
    f"N={int(kan_min_row['hidden_units'])}, "
    f"G={int(kan_min_row['grid_size'])}, "
    f"k={int(kan_min_row['spline_order'])})"
)
print(
    "Maximum: "
    f"{kan_max_params:,} parameters "
    f"(L={int(kan_max_row['hidden_layers'])}, "
    f"N={int(kan_max_row['hidden_units'])}, "
    f"G={int(kan_max_row['grid_size'])}, "
    f"k={int(kan_max_row['spline_order'])})"
)
print()

# Find MLP widths whose parameter count lies inside the full KAN range.
mlp_range_rows = []
for depth in kan_depths:
    mlp_rows = []
    for width in mlp_sweep_widths:
        model_mlp, _ = build_models(
            device,
            hidden_layers=depth,
            hidden_units=width,
        )
        _, _, mlp_params = calculate_flops(
            model_mlp,
            input_shape=input_shape,
            print_results=False,
            print_detailed=False,
            output_as_string=False,
        )
        mlp_rows.append((width, int(mlp_params)))

    mlp_in_range = [
        (width, params)
        for width, params in mlp_rows
        if kan_min_params <= params <= kan_max_params
    ]

    if mlp_in_range:
        first_width, first_params = mlp_in_range[0]
        last_width, last_params = mlp_in_range[-1]
        width_range = f"{first_width}-{last_width}"
        parameter_range = f"{first_params:,}-{last_params:,}"
    else:
        width_range = "No width in tested range"
        parameter_range = "-"

    mlp_range_rows.append({
        "MLP hidden layers": depth,
        "MLP widths in KAN range": width_range,
        "MLP parameter range": parameter_range,
    })

mlp_range_df = pd.DataFrame(mlp_range_rows)
print("MLP WIDTHS WITHIN THE KAN PARAMETER RANGE")
print("=" * 70)
print(mlp_range_df.to_string(index=False))

print()
print(
    "Use the width range above for each MLP depth; the closest individual "
    "match for every KAN architecture is shown in the previous table."
)

KAN PARAMETER RANGE
Minimum: 315 parameters (L=1, N=15, G=3, k=2)
Maximum: 33,215 parameters (L=3, N=35, G=7, k=4)

MLP WIDTHS WITHIN THE KAN PARAMETER RANGE
 MLP hidden layers MLP widths in KAN range MLP parameter range
                 1                  16-149          337-22,947
                 2                  12-127          361-33,021
                 3                  10-104          371-33,177

Use the width range above for each MLP depth; the closest individual match for every KAN architecture is shown in the previous table.


## 4. Recommend Three MLP Configurations

The final recommendations deliberately use one, two, and three hidden layers for the minimum, intermediate, and maximum KAN budgets, respectively.

In [10]:
# Recommend one MLP depth for each KAN budget: 1, 2, and 3 layers.
mlp_candidates = []
for depth in kan_depths:
    for width in mlp_sweep_widths:
        model_mlp, _ = build_models(
            device,
            hidden_layers=depth,
            hidden_units=width,
        )
        _, _, mlp_params = calculate_flops(
            model_mlp,
            input_shape=input_shape,
            print_results=False,
            print_detailed=False,
            output_as_string=False,
        )
        mlp_candidates.append({
            "hidden_layers": depth,
            "hidden_units": width,
            "parameters": int(mlp_params),
        })

mlp_candidates_df = pd.DataFrame(mlp_candidates)
intermediate_params = (kan_min_params + kan_max_params) / 2
targets = [
    ("Minimum KAN budget", kan_min_params, 1),
    ("Intermediate budget", intermediate_params, 2),
    ("Maximum KAN budget", kan_max_params, 3),
]

recommendations = []
for label, target, depth in targets:
    depth_candidates = mlp_candidates_df[
        mlp_candidates_df["hidden_layers"] == depth
    ]
    distances = (depth_candidates["parameters"] - target).abs()
    match = depth_candidates.loc[distances.idxmin()]
    recommendations.append({
        "Target": label,
        "Target parameters": round(target),
        "MLP hidden layers": depth,
        "MLP hidden units": int(match["hidden_units"]),
        "MLP parameters": int(match["parameters"]),
        "Absolute difference": int(abs(match["parameters"] - target)),
    })

recommendations_df = pd.DataFrame(recommendations)
print("RECOMMENDED MLP CONFIGURATIONS")
print("=" * 80)
print(recommendations_df.to_string(index=False))
print()
print(
    "The minimum, intermediate, and maximum budgets are matched to "
    "1-, 2-, and 3-hidden-layer MLPs, respectively."
)

RECOMMENDED MLP CONFIGURATIONS
             Target  Target parameters  MLP hidden layers  MLP hidden units  MLP parameters  Absolute difference
 Minimum KAN budget                315                  1                15             301                   14
Intermediate budget              16765                  2                90           16741                   24
 Maximum KAN budget              33215                  3               104           33177                   38

The minimum, intermediate, and maximum budgets are matched to 1-, 2-, and 3-hidden-layer MLPs, respectively.
